# Run SFT

Connect a Colab **T4** runtime. The kernel folder is `/content` on the Colab VM, not your PC. Before the first cell, upload `train_sft.py`, `common.py`, and the `starter` folder to that server (right-click, Upload to Colab). Training code stays in `train_sft.py`.

The first cell mounts Drive and points `checkpoints/`, `data_cache/`, and `results/` at `MyDrive/rlhf/` so those outputs persist. Optional: put `data.pkl` in `MyDrive/rlhf/data_cache/` before training to skip tokenizing.


In [ ]:
from pathlib import Path
import shutil

from google.colab import drive
drive.mount("/content/drive")

!pip install -q datasets rouge-score matplotlib

root = Path.cwd()
assert (root / "train_sft.py").is_file(), (
    f"train_sft.py is not in {root}. This kernel is the Colab VM. Upload train_sft.py, common.py, and starter/ to the server, then run this cell again."
)

drive_root = Path("/content/drive/MyDrive/rlhf")
for name in ("checkpoints", "data_cache", "results"):
    target = drive_root / name
    target.mkdir(parents=True, exist_ok=True)
    link = root / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        for path in link.rglob("*"):
            if path.is_file():
                dest = target / path.relative_to(link)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(path, dest)
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)

print("project:", root)
print("data cache:", (root / "data_cache" / "data.pkl").exists())
!nvidia-smi -L


In [ ]:
!python train_sft.py


In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("results/sft_samples.md", encoding="utf-8").read()))
display(Image("results/fig1_sft_loss.png"))
